## 1. Import Libraries and Define Global Variables

In [ ]:
import os, gc, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings('ignore')
t0 = time.time()

TARGET = '임신 성공 여부'
ID_COL = 'ID'

age_map = {'만18-34세':26, '만35-37세':36, '만38-39세':38.5, '만40-42세':41,
           '만43-44세':43.5, '만45-50세':47.5, '알 수 없음':np.nan}
donor_age_map = {'만20세 이하':19, '만21-25세':23, '만26-30세':28, '만31-35세':33,
                 '만36-40세':38, '만41-45세':43, '알 수 없음':np.nan}
count_cols = ['총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
              '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
              '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수']
proc_tokens = ['ICSI', 'IVF', 'BLASTOCYST', 'AH', 'Unknown', 'FER']
reasons = ['기증용', '난자 저장용', '배아 저장용', '현재 시술용']
_count_map = {'0회':0.0, '1회':1.0, '2회':2.0, '3회':3.0, '4회':4.0, '5회':5.0, '6회 이상':6.0}

probe = pd.read_csv('train.csv', nrows=200)
obj_cols_src = [c for c in probe.select_dtypes(include='object').columns if c != ID_COL]
dtype_dict = {}
for c in probe.columns:
    if c == ID_COL: continue
    dtype_dict[c] = 'object' if c in obj_cols_src else 'float32'

KEEP_CAT_COLS = ['시술 시기 코드', '시술 당시 나이', '시술 유형',
                 '배란 유도 유형', '난자 출처', '정자 출처',
                 '난자 기증자 나이', '정자 기증자 나이']
DROP_AFTER_FE = ['특정 시술 유형', '배아 생성 주요 이유']


: 

## 2. Feature Engineering Functions

In [ ]:
def process_chunk(df):
    """V2와 동일한 의미의 FE (memory footprint 동일 유지)"""
    for c in count_cols:
        if c in df.columns:
            df[c] = df[c].astype(str).replace('nan', np.nan).map(_count_map).astype('float32')
    df['age_num'] = df['시술 당시 나이'].map(age_map).astype('float32')
    df['egg_donor_age_num'] = df['난자 기증자 나이'].map(donor_age_map).astype('float32')
    df['sperm_donor_age_num'] = df['정자 기증자 나이'].map(donor_age_map).astype('float32')
    s = df['특정 시술 유형'].fillna('').astype(str)
    for tok in proc_tokens:
        df[f'has_{tok}'] = s.str.contains(tok, regex=False).astype('int8')
    s2 = df['배아 생성 주요 이유'].fillna('').astype(str)
    for r in reasons:
        df[f'reason_{r}'] = s2.str.contains(r, regex=False).astype('int8')
    df = df.drop(columns=DROP_AFTER_FE)
    eps = 1e-6
    df['past_preg_rate'] = (df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)).astype('float32')
    df['past_birth_rate'] = (df['총 출산 횟수'] / (df['총 임신 횟수'] + eps)).astype('float32')
    df['ivf_preg_rate'] = (df['IVF 임신 횟수'] / (df['IVF 시술 횟수'] + eps)).astype('float32')
    df['embryo_transfer_ratio'] = (df['이식된 배아 수'] / (df['총 생성 배아 수'] + eps)).astype('float32')
    df['embryo_stored_ratio'] = (df['저장된 배아 수'] / (df['총 생성 배아 수'] + eps)).astype('float32')
    df['icsi_ratio']           = (df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)).astype('float32')
    df['age_x_transferred']    = (df['age_num'] * df['이식된 배아 수']).astype('float32')
    df['gap_transfer_minus_pickup'] = (df['배아 이식 경과일'] - df['난자 채취 경과일']).astype('float32')
    for c in ['착상 전 유전 검사 사용 여부', '임신 시도 또는 마지막 임신 경과 연수', '배아 해동 경과일']:
        if c in df.columns:
            df[f'{c}__isnan'] = df[c].isna().astype('int8')
    return df

def add_v3_features(df):
    """V3 신규 피처 5개를 load 완료된 df에 일괄 추가 (메모리 안전)"""
    # [V3-1] 배양 기간
    df['days_mix_to_transfer'] = (df['배아 이식 경과일'] - df['난자 혼합 경과일']).astype('float32')
    # [V3-2] real_age_num
    is_donor = (df['난자 출처'].astype(str) == '기증 제공')
    df['real_age_num'] = df['age_num'].astype('float32').values.copy()
    df.loc[is_donor, 'real_age_num'] = df.loc[is_donor, 'egg_donor_age_num']
    df['real_age_num'] = df['real_age_num'].astype('float32')
    # [V3-3] log
    df['log_n_embryo'] = np.log1p(df['총 생성 배아 수'].fillna(0)).astype('float32')
    # [V3-4] 유전검사 통합 결측
    df['miss_gene_test'] = (
        df['PGS 시술 여부'].isna() | df['PGD 시술 여부'].isna() | df['착상 전 유전 검사 사용 여부'].isna()
    ).astype('int8')
    # [V3-5] 교호
    df['age_x_embryo_ratio'] = (df['age_num'] * df['embryo_transfer_ratio']).astype('float32')
    return df

def load_chunked(path, chunksize=20000):
    parts = []
    for ch in pd.read_csv(path, dtype=dtype_dict, chunksize=chunksize):
        parts.append(process_chunk(ch)); gc.collect()
    df = pd.concat(parts, ignore_index=True); del parts; gc.collect(); return df

## 3. Load and Preprocess Training Data

In [ ]:
print('[1] Load train ...')
train = load_chunked('train.csv')
y = train[TARGET].astype(np.int8).values
train_ids = train[ID_COL].values
train = train.drop(columns=[TARGET, ID_COL])
print(f' raw-FE train mem={train.memory_usage(deep=True).sum()/1e6:.1f}MB t={time.time()-t0:.1f}s')

train = add_v3_features(train)
gc.collect()

cat_cols = KEEP_CAT_COLS[:]
cat_maps = {}
for c in cat_cols:
    train[c] = train[c].astype(str).replace({'nan': np.nan, 'None': np.nan})
    vals = sorted([v for v in train[c].dropna().unique()])
    cat_maps[c] = vals
    train[c] = pd.Categorical(train[c], categories=vals)
print(f' train {train.shape} mem={train.memory_usage(deep=True).sum()/1e6:.1f}MB')


## 4. Load and Preprocess Test Data

In [ ]:
print('[2] Load test...')
test = load_chunked('test.csv')
test_ids = test[ID_COL].values
test = test.drop(columns=[ID_COL]) # Remove TARGET from columns to drop
test = add_v3_features(test)
for c in cat_cols:
    test[c] = test[c].astype(str).replace({'nan': np.nan, 'None': np.nan})
    vals = cat_maps[c]
    unseen = (~test[c].isin(vals)) & test[c].notna()
    if unseen.any():
        test.loc[unseen, c] = np.nan
    test[c] = pd.Categorical(test[c], categories=vals)
assert list(train.columns) == list(test.columns)
features = train.columns.tolist()
print(f' test mem={test.memory_usage(deep=True).sum()/1e6:.1f}MB # features = {len(features)} (V2=89, +{len(features)-89})')
gc.collect()


## 5. LightGBM Model Training (5-Seed x 5-Fold Cross-Validation)

In [ ]:
N_FOLDS = 5
SEEDS = [42, 2024, 777, 31415, 99999]
oof_all = np.zeros(len(train), dtype=np.float32)
pred_all = np.zeros(len(test), dtype=np.float32)
n_runs = 0

base_params = dict(
    objective='binary', metric='auc',
    learning_rate=0.025,
    num_leaves=95, max_depth=-1,
    feature_fraction=0.8, bagging_fraction=0.85, bagging_freq=1,
    min_data_in_leaf=150, lambda_l1=0.05, lambda_l2=1.0,
    verbose=-1, n_jobs=-1,
)

for sd in SEEDS:
    print(f'\n[seed {sd}] =====')
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
    oof_seed = np.zeros(len(train), dtype=np.float32)
    pred_seed = np.zeros(len(test), dtype=np.float32)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
        t_f = time.time()
        params = {**base_params,
                  'random_state': sd, 'seed': sd,
                  'bagging_seed': sd, 'feature_fraction_seed': sd}
        dtr = lgb.Dataset(train.iloc[tr_idx], label=y[tr_idx],
                          categorical_feature=cat_cols, free_raw_data=True)
        dva = lgb.Dataset(train.iloc[va_idx], label=y[va_idx],
                          categorical_feature=cat_cols, reference=dtr, free_raw_data=True)

        model = lgb.train(params, dtr, num_boost_round=5000,
                          valid_sets=[dva], valid_names=['v'],
                          callbacks=[lgb.early_stopping(120), lgb.log_evaluation(0)])

        oof_seed[va_idx] = model.predict(train.iloc[va_idx], num_iteration=model.best_iteration)
        pred_seed += model.predict(test, num_iteration=model.best_iteration) / N_FOLDS

        auc_f = roc_auc_score(y[va_idx], oof_seed[va_idx])
        print(f'  fold{fold+1} AUC={auc_f:.5f} iter={model.best_iteration} t={time.time()-t_f:.1f}s')

        if sd == 42 and fold == 0:
            fi = pd.DataFrame({'feature': features,
                               'gain': model.feature_importance(importance_type='gain')})
            fi = fi.sort_values('gain', ascending=False)
            fi.to_csv('feature_importance_self_v3.csv', index=False)

        del model, dtr, dva
        gc.collect()

    auc_seed = roc_auc_score(y, oof_seed)
    print(f' [seed{sd}] OOF = {auc_seed:.5f}')
    oof_all += oof_seed
    pred_all += pred_seed
    n_runs += 1
    del oof_seed, pred_seed
    gc.collect()

oof_all /= n_runs
pred_all /= n_runs
auc_final = roc_auc_score(y, oof_all)
print(f'\n[FINAL] Self-V3 OOF AUC = {auc_final:.5f}')


## 6. Ensemble and Comparison with V2 Model

In [ ]:
# The following lines are commented out because the V2 prediction files are missing.
# v2_oof = np.load('oof_lgb_v2_avg.npy')
# v2_pred = np.load('pred_lgb_v2_avg.npy')
# v2_auc = roc_auc_score(y, v2_oof)
# print(f'\n[Compare] V2 = {v2_auc:.5f}   V3 = {auc_final:.5f}   Δ = {auc_final - v2_auc:+.5f}')

# from scipy.stats import rankdata
# oof_avg = (oof_all + v2_oof) / 2
# pred_avg = (pred_all + v2_pred) / 2
# auc_avg = roc_auc_score(y, oof_avg)

# r3_o = rankdata(oof_all) / len(oof_all)
# r2_o = rankdata(v2_oof) / len(v2_oof)
# r3_p = rankdata(pred_all) / len(pred_all)
# r2_p = rankdata(v2_pred) / len(v2_pred)
# oof_rank = (r3_o + r2_o) / 2
# pred_rank = (r3_p + r2_p) / 2
# auc_rank = roc_auc_score(y, oof_rank)

# best_w, best_auc_w = 0.5, 0.0
# for w in np.arange(0.05, 1.0, 0.05):
#     a = roc_auc_score(y, w * oof_all + (1 - w) * v2_oof)
#     if a > best_auc_w:
#         best_auc_w, best_w = a, w
# pred_weighted = best_w * pred_all + (1 - best_w) * v2_pred

print(f'\n[Ensemble]')
print(f' V3 only              : {auc_final:.5f}')
# print(f' V2 only              : {v2_auc:.5f}')
# print(f' Simple Avg           : {auc_avg:.5f}')
# print(f' Rank Avg             : {auc_rank:.5f}')
# print(f' Weighted(w_v3={best_w:.2f}) : {best_auc_w:.5f}')

candidates = {
    'v3_only':      (auc_final, pred_all),
    # 'v2_only':      (v2_auc, v2_pred),
    # 'simple_avg':   (auc_avg, pred_avg),
    # 'rank_avg':     (auc_rank, pred_rank),
    # f'weighted_w{best_w:.2f}': (best_auc_w, pred_weighted),
}
best_name = max(candidates, key=lambda k: candidates[k][0])
best_auc, best_pred = candidates[best_name]
print(f'\n[BEST] {best_name} AUC = {best_auc:.5f}')


In [ ]:

"""
07. Self-made V3 - V2(성공 검증된 구조)에 신규 피처 5개만 조심스럽게 추가
(대회 규정 준수: 타 팀/타 대회 코드 무단 재사용 없음)

V2 -> V3 신규 피처 (5개, 우리 독자 근거):
    [V3-1] days_mix_to_transfer    - 명세서 상 '경과일'들의 차이 (배양 기간)
    [V3-2] real_age_num             - 본인 난자 vs 기증 난자 실제 나이 (데이터 구조 관찰)
    [V3-3] log_n_embryno             - 총 생성 배아 수 log 변환 (분포 교정)
    [V3-4] miss_gene_test           - PGS/PGD/유전검사 통합 결측 플래그
    [V3-5] age_x_embryo_ratio       - 나이 x 이식비율 교호 (V2 SHAP 분석 기반)

시드 확장: V2(3) -> V3(5)
학습률: V2(0.03) -> V3(0.025) - 더 정교한 탐색
"""

import os, gc, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings('ignore')
t0 = time.time()

TARGET = '임신 성공 여부'
ID_COL = 'ID'

age_map = {'만18-34세':26, '만35-37세':36, '만38-39세':38.5, '만40-42세':41,
           '만43-44세':43.5, '만45-50세':47.5, '알 수 없음':np.nan}
donor_age_map = {'만20세 이하':19, '만21-25세':23, '만26-30세':28, '만31-35세':33,
                 '만36-40세':38, '만41-45세':43, '알 수 없음':np.nan}
count_cols = ['총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수',
              '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
              '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수']
proc_tokens = ['ICSI', 'IVF', 'BLASTOCYST', 'AH', 'Unknown', 'FER']
reasons = ['기증용', '난자 저장용', '배아 저장용', '현재 시술용']
_count_map = {'0회':0.0, '1회':1.0, '2회':2.0, '3회':3.0, '4회':4.0, '5회':5.0, '6회 이상':6.0}

probe = pd.read_csv('train.csv', nrows=200)
obj_cols_src = [c for c in probe.select_dtypes(include='object').columns if c != ID_COL]
dtype_dict = {}
for c in probe.columns:
    if c == ID_COL: continue
    dtype_dict[c] = 'object' if c in obj_cols_src else 'float32'

KEEP_CAT_COLS = ['시술 시기 코드', '시술 당시 나이', '시술 유형',
                 '배란 유도 유형', '난자 출처', '정자 출처',
                 '난자 기증자 나이', '정자 기증자 나이']
DROP_AFTER_FE = ['특정 시술 유형', '배아 생성 주요 이유']

def process_chunk(df):
    """V2와 동일한 의미의 FE (memory footprint 동일 유지)"""
    for c in count_cols:
        if c in df.columns:
            df[c] = df[c].astype(str).replace('nan', np.nan).map(_count_map).astype('float32')
    df['age_num'] = df['시술 당시 나이'].map(age_map).astype('float32')
    df['egg_donor_age_num'] = df['난자 기증자 나이'].map(donor_age_map).astype('float32')
    df['sperm_donor_age_num'] = df['정자 기증자 나이'].map(donor_age_map).astype('float32')
    s = df['특정 시술 유형'].fillna('').astype(str)
    for tok in proc_tokens:
        df[f'has_{tok}'] = s.str.contains(tok, regex=False).astype('int8')
    s2 = df['배아 생성 주요 이유'].fillna('').astype(str)
    for r in reasons:
        df[f'reason_{r}'] = s2.str.contains(r, regex=False).astype('int8')
    df = df.drop(columns=DROP_AFTER_FE)
    eps = 1e-6
    df['past_preg_rate'] = (df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)).astype('float32')
    df['past_birth_rate'] = (df['총 출산 횟수'] / (df['총 임신 횟수'] + eps)).astype('float32')
    df['ivf_preg_rate'] = (df['IVF 임신 횟수'] / (df['IVF 시술 횟수'] + eps)).astype('float32')
    df['embryo_transfer_ratio'] = (df['이식된 배아 수'] / (df['총 생성 배아 수'] + eps)).astype('float32')
    df['embryo_stored_ratio'] = (df['저장된 배아 수'] / (df['총 생성 배아 수'] + eps)).astype('float32')
    df['icsi_ratio']           = (df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)).astype('float32')
    df['age_x_transferred']    = (df['age_num'] * df['이식된 배아 수']).astype('float32')
    df['gap_transfer_minus_pickup'] = (df['배아 이식 경과일'] - df['난자 채취 경과일']).astype('float32')
    for c in ['착상 전 유전 검사 사용 여부', '임신 시도 또는 마지막 임신 경과 연수', '배아 해동 경과일']:
        if c in df.columns:
            df[f'{c}__isnan'] = df[c].isna().astype('int8')
    return df

def add_v3_features(df):
    """V3 신규 피처 5개를 load 완료된 df에 일괄 추가 (메모리 안전)"""
    # [V3-1] 배양 기간
    df['days_mix_to_transfer'] = (df['배아 이식 경과일'] - df['난자 혼합 경과일']).astype('float32')
    # [V3-2] real_age_num
    is_donor = (df['난자 출처'].astype(str) == '기증 제공')
    df['real_age_num'] = df['age_num'].astype('float32').values.copy()
    df.loc[is_donor, 'real_age_num'] = df.loc[is_donor, 'egg_donor_age_num']
    df['real_age_num'] = df['real_age_num'].astype('float32')
    # [V3-3] log
    df['log_n_embryo'] = np.log1p(df['총 생성 배아 수'].fillna(0)).astype('float32')
    # [V3-4] 유전검사 통합 결측
    df['miss_gene_test'] = (
        df['PGS 시술 여부'].isna() | df['PGD 시술 여부'].isna() | df['착상 전 유전 검사 사용 여부'].isna()
    ).astype('int8')
    # [V3-5] 교호
    df['age_x_embryo_ratio'] = (df['age_num'] * df['embryo_transfer_ratio']).astype('float32')
    return df

def load_chunked(path, chunksize=20000):
    parts = []
    for ch in pd.read_csv(path, dtype=dtype_dict, chunksize=chunksize):
        parts.append(process_chunk(ch)); gc.collect()
    df = pd.concat(parts, ignore_index=True); del parts; gc.collect(); return df

print('[1] Load train (V2와 동일 load)...')
train = load_chunked('train.csv')
y = train[TARGET].astype(np.int8).values
train_ids = train[ID_COL].values
train = train.drop(columns=[TARGET, ID_COL])
print(f' raw-FE train mem={train.memory_usage(deep=True).sum()/1e6:.1f}MB t={time.time()-t0:.1f}s')

# V3 신규 피처 추가 (load 완료 후에 메모리 안전하게)
train = add_v3_features(train)
gc.collect()

cat_cols = KEEP_CAT_COLS[:]
cat_maps = {}
for c in cat_cols:
    train[c] = train[c].astype(str).replace({'nan': np.nan, 'None': np.nan})
    vals = sorted([v for v in train[c].dropna().unique()])
    cat_maps[c] = vals
    train[c] = pd.Categorical(train[c], categories=vals)
print(f' train {train.shape} mem={train.memory_usage(deep=True).sum()/1e6:.1f}MB')

print('[2] Load test...')
test = load_chunked('test.csv')
test_ids = test[ID_COL].values
test = test.drop(columns=[ID_COL]) # Removed TARGET from columns to drop
test = add_v3_features(test)
for c in cat_cols:
    test[c] = test[c].astype(str).replace({'nan': np.nan, 'None': np.nan})
    vals = cat_maps[c]
    unseen = (~test[c].isin(vals)) & test[c].notna()
    if unseen.any():
        test.loc[unseen, c] = np.nan
    test[c] = pd.Categorical(test[c], categories=vals)
assert list(train.columns) == list(test.columns)
features = train.columns.tolist()
print(f' test mem={test.memory_usage(deep=True).sum()/1e6:.1f}MB # features = {len(features)} (V2=89, +{len(features)-89})')
gc.collect()

# ---------- 5-Seed x 5-Fold ----------
N_FOLDS = 5
SEEDS = [42, 2024, 777, 31415, 99999]  # V2(3) -> V3(5 seeds)
oof_all = np.zeros(len(train), dtype=np.float32)
pred_all = np.zeros(len(test), dtype=np.float32)
n_runs = 0

base_params = dict(
    objective='binary', metric='auc',
    learning_rate=0.025,           # V2=0.03 -> 더 정교한 탐색
    num_leaves=95, max_depth=-1,
    feature_fraction=0.8, bagging_fraction=0.85, bagging_freq=1,
    min_data_in_leaf=150, lambda_l1=0.05, lambda_l2=1.0,
    verbose=-1, n_jobs=-1,
)

for sd in SEEDS:
    print(f'\n[seed {sd}] =====')
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=sd)
    oof_seed = np.zeros(len(train), dtype=np.float32)
    pred_seed = np.zeros(len(test), dtype=np.float32)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
        t_f = time.time()
        params = {**base_params,
                  'random_state': sd, 'seed': sd,
                  'bagging_seed': sd, 'feature_fraction_seed': sd}
        dtr = lgb.Dataset(train.iloc[tr_idx], label=y[tr_idx],
                          categorical_feature=cat_cols, free_raw_data=True)
        dva = lgb.Dataset(train.iloc[va_idx], label=y[va_idx],
                          categorical_feature=cat_cols, reference=dtr, free_raw_data=True)

        model = lgb.train(params, dtr, num_boost_round=5000,
                          valid_sets=[dva], valid_names=['v'],
                          callbacks=[lgb.early_stopping(120), lgb.log_evaluation(0)])

        oof_seed[va_idx] = model.predict(train.iloc[va_idx], num_iteration=model.best_iteration)
        pred_seed += model.predict(test, num_iteration=model.best_iteration) / N_FOLDS

        auc_f = roc_auc_score(y[va_idx], oof_seed[va_idx])
        print(f'  fold{fold+1} AUC={auc_f:.5f} iter={model.best_iteration} t={time.time()-t_f:.1f}s')

        if sd == 42 and fold == 0:
            fi = pd.DataFrame({'feature': features,
                               'gain': model.feature_importance(importance_type='gain')})
            fi = fi.sort_values('gain', ascending=False)
            fi.to_csv('feature_importance_self_v3.csv', index=False)

        del model, dtr, dva
        gc.collect()

    auc_seed = roc_auc_score(y, oof_seed)
    print(f' [seed{sd}] OOF = {auc_seed:.5f}')
    oof_all += oof_seed
    pred_all += pred_seed
    n_runs += 1
    del oof_seed, pred_seed
    gc.collect()

oof_all /= n_runs
pred_all /= n_runs
auc_final = roc_auc_score(y, oof_all)
print(f'\n[FINAL] Self-V3 OOF AUC = {auc_final:.5f}')

# ---------- 비교 & 앙상블 ----------
# v2_oof = np.load('oof_lgb_v2_avg.npy')
# v2_pred = np.load('pred_lgb_v2_avg.npy')
# v2_auc = roc_auc_score(y, v2_oof)
# print(f'\n[Compare] V2 = {v2_auc:.5f}   V3 = {auc_final:.5f}   Δ = {auc_final - v2_auc:+.5f}')

# from scipy.stats import rankdata
# oof_avg = (oof_all + v2_oof) / 2
# pred_avg = (pred_all + v2_pred) / 2
# auc_avg = roc_auc_score(y, oof_avg)

# r3_o = rankdata(oof_all) / len(oof_all)
# r2_o = rankdata(v2_oof) / len(v2_oof)
# r3_p = rankdata(pred_all) / len(pred_all)
# r2_p = rankdata(v2_pred) / len(v2_pred)
# oof_rank = (r3_o + r2_o) / 2
# pred_rank = (r3_p + r2_p) / 2
# auc_rank = roc_auc_score(y, oof_rank)

# best_w, best_auc_w = 0.5, 0.0
# for w in np.arange(0.05, 1.0, 0.05):
#     a = roc_auc_score(y, w * oof_all + (1 - w) * v2_oof)
#     if a > best_auc_w:
#         best_auc_w, best_w = a, w
# pred_weighted = best_w * pred_all + (1 - best_w) * v2_pred

print(f'\n[Ensemble]')
print(f' V3 only              : {auc_final:.5f}')
# print(f' V2 only              : {v2_auc:.5f}')
# print(f' Simple Avg           : {auc_avg:.5f}')
# print(f' Rank Avg             : {auc_rank:.5f}')
# print(f' Weighted(w_v3={best_w:.2f}) : {best_auc_w:.5f}')

candidates = {
    'v3_only':      (auc_final, pred_all),
    # 'v2_only':      (v2_auc, v2_pred),
    # 'simple_avg':   (auc_avg, pred_avg),
    # 'rank_avg':     (auc_rank, pred_rank),
    # f'weighted_w{best_w:.2f}': (best_auc_w, pred_weighted),
}
best_name = max(candidates, key=lambda k: candidates[k][0])
best_auc, best_pred = candidates[best_name]
print(f'\n[BEST] {best_name} AUC = {best_auc:.5f}')

np.save('oof_self_v3.npy', oof_all)
np.save('pred_self_v3.npy', pred_all)

sub = pd.read_csv('sample_submission.csv')
sub_df = pd.DataFrame({'ID': test_ids, 'probability': np.clip(best_pred, 0, 1)})
sub = sub[['ID']].merge(sub_df, on='ID', how='left')
out_path = f'submission_self_v3_{best_name}_{best_auc:.5f}.csv'
sub.to_csv(out_path, index=False)
print(f'[Saved] {out_path}')
print(sub.head())
print(sub['probability'].describe())
print(f'\n[Done] {time.time()-t0:.1f}s')
